# ???????? Faster R-CNN ??? ???????? ???????? ?? ???

??????? ??? Google Colab. ?????????????? ???????, ??????? Faster R-CNN ? ????????? ????, ??????? ? ??????? ?? Google Drive.


In [ ]:
!pip install -q kagglehub torchmetrics pycocotools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_NAME = 'brain-mri-faster-rcnn'
WORKDIR = Path('/content') / PROJECT_NAME
ARTIFACTS_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
DATASET_DIR = WORKDIR / 'dataset'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
RUNS_DIR = ARTIFACTS_DIR / 'runs'

EPOCHS = 10
BATCH_SIZE = 4
NUM_WORKERS = 2
LEARNING_RATE = 0.005
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9
SEED = 42
SCORE_THRESHOLD = 0.25

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
NUM_CLASSES = len(CLASS_NAMES) + 1

WORKDIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR:', WORKDIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)

In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

In [ ]:
import kagglehub
import shutil
from pathlib import Path

path = kagglehub.dataset_download('ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes')
dataset_path = Path(path)
print('Raw kagglehub path:', dataset_path)

candidate_roots = [dataset_path] + [p for p in dataset_path.rglob('*') if p.is_dir()]
resolved_root = None
for candidate in candidate_roots:
    if (candidate / 'Train').exists() and (candidate / 'Val').exists():
        resolved_root = candidate
        break

if resolved_root is None:
    raise FileNotFoundError('Не удалось найти папки Train/Val внутри датасета.')

dataset_target = DATASET_DIR / 'brain-tumor-mri'
if dataset_target.exists():
    shutil.rmtree(dataset_target)
shutil.copytree(resolved_root, dataset_target)

print('Resolved dataset root:', resolved_root)
print('Copied dataset to:', dataset_target)

In [ ]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

class BrainMRIDetectionDataset(Dataset):
    def __init__(self, dataset_root, split_name, class_names):
        self.dataset_root = Path(dataset_root)
        self.split_name = split_name
        self.class_names = class_names
        self.records = []
        split_dir = self.dataset_root / split_name

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            for image_path in sorted((class_dir / 'images').glob('*')):
                label_path = class_dir / 'labels' / f'{image_path.stem}.txt'
                self.records.append({
                    'image_path': image_path,
                    'label_path': label_path,
                })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record['image_path']).convert('RGB')
        width, height = image.size

        boxes = []
        labels = []

        if record['label_path'].exists():
            content = record['label_path'].read_text(encoding='utf-8').strip().splitlines()
            for line in content:
                if not line.strip():
                    continue
                class_id, x_center, y_center, box_width, box_height = map(float, line.split())
                x1 = (x_center - box_width / 2.0) * width
                y1 = (y_center - box_height / 2.0) * height
                x2 = (x_center + box_width / 2.0) * width
                y2 = (y_center + box_height / 2.0) * height
                boxes.append([x1, y1, x2, y2])
                labels.append(int(class_id) + 1)

        image_tensor = F.to_tensor(image)

        if boxes:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
            area = (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * (boxes_tensor[:, 3] - boxes_tensor[:, 1])
        else:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)

        target = {
            'boxes': boxes_tensor,
            'labels': labels_tensor,
            'image_id': torch.tensor([index]),
            'area': area,
            'iscrowd': torch.zeros((labels_tensor.shape[0],), dtype=torch.int64),
        }

        return image_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

DATA_ROOT = DATASET_DIR / 'brain-tumor-mri'
train_dataset = BrainMRIDetectionDataset(DATA_ROOT, 'Train', CLASS_NAMES)
val_dataset = BrainMRIDetectionDataset(DATA_ROOT, 'Val', CLASS_NAMES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print('Train samples:', len(train_dataset))
print('Val samples:', len(val_dataset))

In [ ]:
from collections import Counter
import json

def collect_split_summary(split_name):
    split_dir = DATA_ROOT / split_name
    image_counts = Counter()
    box_counts = Counter()
    missing_labels = 0
    empty_labels = 0
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        images = sorted((class_dir / 'images').glob('*'))
        image_counts[class_dir.name] = len(images)
        for image_path in images:
            label_path = class_dir / 'labels' / f'{image_path.stem}.txt'
            if not label_path.exists():
                missing_labels += 1
                continue
            lines = label_path.read_text(encoding='utf-8').strip().splitlines()
            if not lines:
                empty_labels += 1
            box_counts[class_dir.name] += len(lines)
    return {
        'images_per_class': dict(image_counts),
        'boxes_per_class': dict(box_counts),
        'missing_labels': missing_labels,
        'empty_labels': empty_labels,
    }

dataset_summary = {
    'train': collect_split_summary('Train'),
    'val': collect_split_summary('Val'),
}

summary_path = ARTIFACTS_DIR / 'dataset_summary.json'
summary_path.write_text(json.dumps(dataset_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_model(NUM_CLASSES)
model.to(DEVICE)
model

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

history = []
best_map50 = -1.0
best_weights_path = RUNS_DIR / 'best_faster_rcnn.pt'
last_weights_path = RUNS_DIR / 'last_faster_rcnn.pt'

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(loader, desc='Train', leave=False):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        running_loss += losses.item()
    return running_loss / max(len(loader), 1)

def evaluate_model(model, loader, device, score_threshold=0.25):
    metric = MeanAveragePrecision(class_metrics=False)
    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Val', leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)

            preds = []
            refs = []
            for output, target in zip(outputs, targets):
                keep = output['scores'].detach().cpu() >= score_threshold
                preds.append({
                    'boxes': output['boxes'].detach().cpu()[keep],
                    'scores': output['scores'].detach().cpu()[keep],
                    'labels': output['labels'].detach().cpu()[keep],
                })
                refs.append({
                    'boxes': target['boxes'].detach().cpu(),
                    'labels': target['labels'].detach().cpu(),
                })
            metric.update(preds, refs)

    computed = metric.compute()
    return {
        'map': float(computed['map']),
        'map_50': float(computed['map_50']),
        'mar_100': float(computed['mar_100']),
    }

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, DEVICE, SCORE_THRESHOLD)
    lr_scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'map': val_metrics['map'],
        'map_50': val_metrics['map_50'],
        'mar_100': val_metrics['mar_100'],
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(row)

    torch.save(model.state_dict(), last_weights_path)
    if row['map_50'] > best_map50:
        best_map50 = row['map_50']
        torch.save(model.state_dict(), best_weights_path)

history_df = pd.DataFrame(history)
history_path = RUNS_DIR / 'metrics.csv'
history_df.to_csv(history_path, index=False)
history_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o')
plt.title('Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['map_50'], marker='o', label='mAP@50')
plt.plot(history_df['epoch'], history_df['map'], marker='o', label='mAP@50:95')
plt.title('Validation Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.tight_layout()

metrics_plot_path = RUNS_DIR / 'training_curves.png'
plt.savefig(metrics_plot_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

best_model = build_model(NUM_CLASSES)
best_model.load_state_dict(torch.load(best_weights_path, map_location=DEVICE))
best_model.to(DEVICE)
best_model.eval()

sample_count = 6
sample_images = [val_dataset.records[i]['image_path'] for i in range(min(sample_count, len(val_dataset.records)))]
preview_dir = PREDICTIONS_DIR / 'val_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(14, 10))
with torch.no_grad():
    for idx, image_path in enumerate(sample_images, start=1):
        image = Image.open(image_path).convert('RGB')
        image_tensor = F.to_tensor(image).to(DEVICE)
        prediction = best_model([image_tensor])[0]

        image_np = np.array(image).copy()
        keep = prediction['scores'].detach().cpu().numpy() >= SCORE_THRESHOLD
        boxes = prediction['boxes'].detach().cpu().numpy()[keep]
        labels = prediction['labels'].detach().cpu().numpy()[keep]
        scores = prediction['scores'].detach().cpu().numpy()[keep]

        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.astype(int)
            class_name = CLASS_NAMES[int(label) - 1]
            cv2.rectangle(image_np, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image_np, f'{class_name} {score:.2f}', (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        output_path = preview_dir / f'image{idx - 1}.jpg'
        Image.fromarray(image_np).save(output_path)

        plt.subplot(2, 3, idx)
        plt.imshow(image_np)
        plt.axis('off')
        plt.title(image_path.name)

plt.tight_layout()
plt.show()

In [ ]:
import shutil
import json

final_dir = ARTIFACTS_DIR / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)

summary_metrics = history_df.iloc[history_df['map_50'].idxmax()].to_dict()
summary_metrics_path = final_dir / 'best_metrics.json'
summary_metrics_path.write_text(json.dumps(summary_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

files_to_copy = [
    best_weights_path,
    last_weights_path,
    history_path,
    metrics_plot_path,
    summary_path,
    summary_metrics_path,
]

for file_path in files_to_copy:
    if file_path.exists():
        shutil.copy2(file_path, final_dir / file_path.name)

print('Artifacts directory:', final_dir)
print('Available files:')
for path in sorted(final_dir.glob('*')):
    print('-', path.name)

## ??? ????????? ????? ????????

????? ?????????? ???????? ??????? ????????? `best_faster_rcnn.pt`, `last_faster_rcnn.pt`, `metrics.csv`, `training_curves.png`, `dataset_summary.json` ? `best_metrics.json`.
